In [1]:
import pandas as pd
import numpy as np


class ProfitabilityAnalyzer:
    """
    Prepare an analysis-ready retail profitability dataset.

    Input workbook must contain:
        - order_lines
        - returns
        - monthly_targets

    Output:
        - analysis_ready_retail.xlsx
    """

    def __init__(self, file_path):
        self.file_path = file_path

        self.order_lines = pd.read_excel(
            file_path,
            sheet_name="order_lines"
        )

        self.returns = pd.read_excel(
            file_path,
            sheet_name="returns"
        )

        self.monthly_targets = pd.read_excel(
            file_path,
            sheet_name="monthly_targets"
        )

        self.analysis_ready = None

    def data_quality_checks(self):
        """Perform only the requested data-quality checks."""

        orders = self.order_lines
        returns = self.returns

        print("\n--- DATA QUALITY CHECKS ---")

        print("Duplicate order_line_id:",
              orders["order_line_id"].duplicated().sum())

        print("Missing order_line_id:",
              orders["order_line_id"].isna().sum())

        print("Quantity <= 0:",
              (orders["quantity"] <= 0).sum())

        print("Invalid unit_price:",
              (orders["unit_price"] <= 0).sum())

        print("Invalid cost_per_unit:",
              (orders["cost_per_unit"] <= 0).sum())

        print("Discount outside 0-100:",
              (~orders["discount_pct"].between(0, 100)).sum())

        # Check return IDs that do not exist in order_lines
        invalid_return_links = (
            ~returns["order_line_id"].isin(
                orders["order_line_id"]
            )
        ).sum()

        print("Returns linked to missing order_line_id:",
              invalid_return_links)

        # Return-date validation
        return_dates = returns.merge(
            orders[
                ["order_line_id", "order_date", "quantity"]
            ],
            on="order_line_id",
            how="left"
        )

        invalid_return_dates = (
            return_dates["return_date"]
            < return_dates["order_date"]
        ).sum()

        print("Return date earlier than order date:",
              invalid_return_dates)

        # Return quantity > order-line quantity
        invalid_return_qty = (
            return_dates["return_qty"]
            > return_dates["quantity"]
        ).sum()

        print("Return quantity > order quantity:",
              invalid_return_qty)

    def prepare_data(self):
        """Clean data, aggregate returns, and calculate profitability."""

        orders = self.order_lines.copy()
        returns = self.returns.copy()

        # Convert dates
        orders["order_date"] = pd.to_datetime(
            orders["order_date"],
            errors="coerce"
        )

        returns["return_date"] = pd.to_datetime(
            returns["return_date"],
            errors="coerce"
        )

        # Remove missing/duplicate order_line_id
        orders = orders.dropna(
            subset=["order_line_id"]
        )

        orders = orders.drop_duplicates(
            subset=["order_line_id"]
        )

        # Keep valid order lines
        orders = orders[
            (orders["quantity"] > 0) &
            (orders["unit_price"] > 0) &
            (orders["cost_per_unit"] > 0) &
            (orders["discount_pct"].between(0, 100))
        ].copy()

        # Keep returns linked to valid order lines
        returns = returns[
            returns["order_line_id"].isin(
                orders["order_line_id"]
            )
        ].copy()

        # Add order quantity/date to validate returns
        returns = returns.merge(
            orders[
                ["order_line_id", "quantity", "order_date"]
            ],
            on="order_line_id",
            how="left"
        )

        # Required return validations
        returns = returns[
            returns["return_qty"].ge(0) &
            returns["return_qty"].le(returns["quantity"]) &
            returns["return_date"].ge(returns["order_date"])
        ].copy()

        # IMPORTANT:
        # Aggregate returns before joining them to order_lines.
        # This prevents duplicate sales/revenue when an order line
        # has multiple return records.
        returns_agg = (
            returns
            .groupby("order_line_id", as_index=False)
            .agg(
                total_return_qty=("return_qty", "sum"),
                total_refund=("refund_amount", "sum")
            )
        )

        # Join one row per order line
        data = orders.merge(
            returns_agg,
            on="order_line_id",
            how="left"
        )

        data["total_return_qty"] = (
            data["total_return_qty"]
            .fillna(0)
        )

        data["total_refund"] = (
            data["total_refund"]
            .fillna(0)
        )

        # Cap aggregated returns at ordered quantity
        data["total_return_qty"] = np.minimum(
            data["total_return_qty"],
            data["quantity"]
        )

        # -----------------------------
        # PROFITABILITY CALCULATIONS
        # -----------------------------

        data["gross_revenue"] = (
            data["quantity"] *
            data["unit_price"]
        )

        data["discount_amount"] = (
            data["gross_revenue"] *
            data["discount_pct"] /
            100
        )

        data["net_revenue"] = (
            data["gross_revenue"] -
            data["discount_amount"]
        )

        data["realized_revenue"] = (
            data["net_revenue"] -
            data["total_refund"]
        )

        data["realized_cogs"] = (
            data["cost_per_unit"] *
            (
                data["quantity"] -
                data["total_return_qty"]
            )
        )

        data["realized_profit"] = (
            data["realized_revenue"] -
            data["realized_cogs"]
        )

        data["realized_margin_pct"] = np.where(
            data["realized_revenue"] != 0,
            data["realized_profit"] /
            data["realized_revenue"] * 100,
            np.nan
        )

        # Discount groups
        data["discount_group"] = np.select(
            [
                data["discount_pct"].between(0, 10),
                (
                    data["discount_pct"].gt(10) &
                    data["discount_pct"].le(25)
                ),
                data["discount_pct"].gt(25)
            ],
            [
                "Low Discount",
                "Medium Discount",
                "High Discount"
            ],
            default="Invalid"
        )

        # Month used by SQL/Power BI
        data["month"] = (
            data["order_date"]
            .dt.to_period("M")
            .dt.to_timestamp()
        )

        self.analysis_ready = data

        return data

    def export(self, output_file="analysis_ready_retail.xlsx"):
        """Export analysis-ready data and the target table."""

        if self.analysis_ready is None:
            self.prepare_data()

        with pd.ExcelWriter(
            output_file,
            engine="openpyxl"
        ) as writer:

            self.analysis_ready.to_excel(
                writer,
                sheet_name="analysis_ready",
                index=False
            )

            self.monthly_targets.to_excel(
                writer,
                sheet_name="monthly_targets",
                index=False
            )

        print(f"\nSaved: {output_file}")


if __name__ == "__main__":

    analyzer = ProfitabilityAnalyzer(
        "profitable_growth_concept_case.xlsx"
    )

    analyzer.data_quality_checks()

    df = analyzer.prepare_data()

    print("\nAnalysis-ready rows:", len(df))
    print("\nFirst 5 rows:")
    print(df.head())

    analyzer.export(
        "analysis_ready_retail.xlsx"
    )



--- DATA QUALITY CHECKS ---
Duplicate order_line_id: 102
Missing order_line_id: 35
Quantity <= 0: 55
Invalid unit_price: 35
Invalid cost_per_unit: 0
Discount outside 0-100: 35
Returns linked to missing order_line_id: 20
Return date earlier than order date: 21
Return quantity > order quantity: 42

Analysis-ready rows: 1927

First 5 rows:
  order_line_id order_id customer_id order_date region      channel  \
0      OL000001  O100000       C1120 2026-03-26  South  Marketplace   
1      OL000002  O100000       C1120 2026-03-26  South  Marketplace   
2      OL000003  O100001       C1078 2026-05-08   West       Online   
3      OL000004  O100002       C1050 2026-04-04   West  Marketplace   
4      OL000005  O100003       C1450 2026-01-22   East       Online   

  product_category product_name  quantity  unit_price  ...  total_refund  \
0             Home  Storage Box         4     1485.64  ...           0.0   
1      Electronics      Earbuds         2     7660.36  ...           0.0   
2    